# NB_lab_test_trends — Gold Layer

Builds `adb_caresync.gold.lab_test_trends`: one row per test type per calendar month. Aggregates lab test counts and abnormal/critical rates, enriches with calendar month name and quarter, and computes month-over-month abnormal rate change via a LAG window. Full rebuild on every run.

In [0]:
%sql
-- Ensure the gold schema exists before writing
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold

In [0]:
# Verify all required source silver tables are accessible
source_tables = [
    "adb_caresync.silver.lab_results",
    "adb_caresync.silver.calendar",
]

for t in source_tables:
    count = spark.table(t).count()
    schema = spark.table(t).dtypes
    print(f"✅ {t}: {count:,} rows | columns: {[c[0] for c in schema]}")

In [0]:
%sql
-- ─────────────────────────────────────────────────────────────
-- Target  : adb_caresync.gold.lab_test_trends
-- Grain   : One row per test_name per calendar month
-- Sources : silver.lab_results (driving, grouped by test_name + year + month)
--           silver.calendar    (LEFT JOIN on first day of month → date_key)
-- Windows : LAG(abnormal_rate_pct) PARTITION BY test_name ORDER BY year, month
-- Load    : FULL — CREATE OR REPLACE on every run
-- ─────────────────────────────────────────────────────────────
CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends AS

WITH base AS (
    -- Step 1: aggregate lab_results per test_name per year-month; enrich via calendar join
    SELECT
        lr.test_name,
        YEAR(lr.test_date)                                                     AS year,
        MONTH(lr.test_date)                                                    AS month,
        cal.month_name,
        CONCAT('Q', cal.quarter)                                               AS quarter_name,
        COUNT(*)                                                               AS total_tests,
        COUNT(CASE WHEN lr.result_status = 'Abnormal' THEN 1 END)             AS abnormal_test_count,
        COUNT(CASE WHEN lr.result_status = 'Critical' THEN 1 END)             AS critical_test_count
    FROM      adb_caresync.silver.lab_results lr
    LEFT JOIN adb_caresync.silver.calendar    cal
           ON CAST(DATE_FORMAT(DATE_TRUNC('month', lr.test_date), 'yyyyMMdd') AS INT) = cal.date_key
    GROUP BY
        lr.test_name,
        YEAR(lr.test_date),
        MONTH(lr.test_date),
        cal.month_name,
        cal.quarter
),

with_rates AS (
    -- Step 2: rate columns + LAG for previous-month abnormal rate
    SELECT
        *,
        CAST(abnormal_test_count / total_tests * 100 AS DECIMAL(5,2))          AS abnormal_rate_pct,
        CAST(critical_test_count / total_tests * 100 AS DECIMAL(5,2))          AS critical_rate_pct,
        LAG(CAST(abnormal_test_count / total_tests * 100 AS DECIMAL(5,2)))
            OVER (
                PARTITION BY test_name
                ORDER BY     year, month
            )                                                                   AS prev_month_abnormal_rate_pct
    FROM base
)

-- Step 3: month-over-month percentage-point change in abnormal rate
SELECT
    test_name,
    year,
    month,
    month_name,
    quarter_name,
    total_tests,
    abnormal_test_count,
    critical_test_count,
    abnormal_rate_pct,
    critical_rate_pct,
    prev_month_abnormal_rate_pct,
    CASE
        WHEN prev_month_abnormal_rate_pct IS NULL THEN NULL
        ELSE CAST(abnormal_rate_pct - prev_month_abnormal_rate_pct AS DECIMAL(6,2))
    END AS abnormal_rate_change_pct
FROM with_rates

In [0]:
record_count = spark.table("adb_caresync.gold.lab_test_trends").count()
dbutils.notebook.exit(str(record_count))